# Python DateTime 처리 실습

본 노트북은 강의자료 `[참고] Python DateTime 처리`의 내용을 코드로 직접 실행하며 학습할 수 있도록 구성되었습니다.

## 목차

- **Part 1.** Python 내장 `datetime` 모듈
- **Part 2.** Pandas 시계열 4대 코어 객체
- **Part 3.** Pandas 시계열 특화 기능 (`date_range`, `resample`, `shift`, `rolling`)
- **Part 4.** `dateutil.relativedelta` — 인간 친화적 의미 연산


In [ ]:
# 공통 라이브러리 임포트
import numpy as np
import pandas as pd
from datetime import date, time, datetime, timedelta
from dateutil.relativedelta import relativedelta

print(f"pandas version: {pd.__version__}")

---

# Part 1. Python 내장 `datetime` 모듈

파이썬 내장 `datetime` 모듈은 4개의 핵심 클래스로 구성됩니다.

| 클래스 | 역할 | 주요 속성 |
|---|---|---|
| `date` | 날짜 정보 전용 | `year`, `month`, `day` |
| `time` | 시간 정보 전용 | `hour`, `minute`, `second`, `microsecond` |
| `datetime` | 날짜 + 시간 결합 | `date`와 `time`의 모든 속성 |
| `timedelta` | 두 시점 간의 차이(기간) | `days`, `seconds`, `microseconds` |


## 1-1. `date`와 `time`: 날짜와 시간 요소의 독립적 제어

In [ ]:
# date 클래스: 연, 월, 일 정보만 취급
today = date.today()
d = date(2026, 4, 10)

print(f"오늘 날짜    : {today}")
print(f"특정 날짜    : {d}")
print(f"연도, 월     : {d.year}, {d.month}")
print(f"일           : {d.day}")

In [ ]:
# time 클래스: 시, 분, 초, 마이크로초 정보만 취급 (날짜 없음)
lunch = time(12, 30, 0)
meet = time(hour=14, minute=30)

print(f"점심 시간    : {lunch}")
print(f"미팅 시간    : {meet}")
print(f"meet.hour, meet.minute → {meet.hour}, {meet.minute}")

## 1-2. `datetime`: 두 요소를 결합한 종합 클래스 (실무 활용도 1위)

로그 기록, 거래 시각, 데이터베이스 Timestamp 저장 등 실무 표준으로 사용됩니다.

In [ ]:
# 1. 현재 일시 캡처 및 특정 일시 생성
now = datetime.now()
meeting = datetime(2026, 4, 15, 14, 30, 0)

print(f"현재 일시    : {now}")
print(f"미팅 일시    : {meeting}")

# 2. 결합된 객체에서 하위 요소 분리 추출
print(f"날짜 부분    : {meeting.date()}")   # 2026-04-15
print(f"시간 부분    : {meeting.time()}")   # 14:30:00

# 3. 국제 표준 규격(ISO 8601) 문자열 변환
print(f"ISO 포맷     : {meeting.isoformat()}")  # 2026-04-15T14:30:00

## 1-3. `timedelta`: 타임라인 전후 이동 연산

두 `datetime` 객체 간의 뺄셈 결과 또는 날짜 이동 연산에 사용합니다.

In [ ]:
now = datetime.now()

# 과거/미래 시점 도출
yesterday = now - timedelta(days=1)
next_week = now + timedelta(weeks=1)

print(f"현재       : {now}")
print(f"어제(-1일) : {yesterday}")
print(f"다음주(+1주): {next_week}")

# 복합 기간 연산
complex_delta = timedelta(days=3, hours=5, minutes=30)
future = now + complex_delta
print(f"3일 5시간 30분 뒤: {future}")

# D-Day 잔여 시간 완벽 계산
deadline = datetime(2026, 12, 31, 23, 59, 59)
remaining = deadline - now
print(f"연말까지 잔여 일수: {remaining.days}일")
print(f"초 단위 정밀 환산: {remaining.total_seconds():,.0f}초")

### ⚠️ `timedelta`의 치명적 한계: 월(Month) / 연(Year) 단위 연산 불가

- **지원 단위**: `days`, `seconds`, `microseconds` 세 가지만 파라미터로 허용
- **발생 문제**: *'정확히 1달 후'*, *'1년 후'*와 같은 연산 불가능
- **원인**: 그레고리력에서 월의 일수(28~31일)와 윤년 여부가 가변적이기 때문
- **해결책**: `dateutil.relativedelta` 사용 필수 (→ Part 4 참조)

In [ ]:
# timedelta로는 'months=1'을 쓸 수 없음 — 아래 주석을 풀면 TypeError 발생
# bad = timedelta(months=1)

try:
    bad = timedelta(months=1)
except TypeError as e:
    print(f"TypeError 발생: {e}")

# 대체로 days=30을 쓰면 '정확한 1개월 후'가 아님
approx_month = datetime(2026, 1, 31) + timedelta(days=30)
print(f"1월 31일 + 30일 = {approx_month.date()}  (→ 2월 말이 아닌 3월 2일)")

## 1-4. [실습] D-Day 연산 엔진

세 단계 파이프라인으로 D-Day 계산기를 구축합니다.

1. **Parsing**: `date.fromisoformat()`로 문자열 파싱
2. **Math**: 타겟 날짜에서 오늘을 빼서 `timedelta` 계산
3. **Routing**: `delta.days` 부호에 따라 분기 처리
    - `delta.days > 0` → 잔여 일수 표시 (예: D-50)
    - `delta.days == 0` → 당일 알림 (D-Day!)
    - `delta.days < 0` → 경과 일수 표시 (예: D+10)

In [ ]:
def dday_engine(target_str: str) -> str:
    """ISO 포맷(YYYY-MM-DD) 문자열을 받아 D-Day 라벨을 반환한다."""
    # Step 1. Parsing
    target = date.fromisoformat(target_str)
    # Step 2. Math
    delta = target - date.today()
    # Step 3. Routing
    if delta.days > 0:
        return f"D-{delta.days}"
    elif delta.days == 0:
        return "D-Day!"
    else:
        return f"D+{abs(delta.days)}"

# 테스트
for target in ["2026-12-31", "2026-01-01", str(date.today())]:
    print(f"{target} → {dday_engine(target)}")

## 1-5. Format vs Parse: `strftime`과 `strptime`

| 함수 | 설명 | 방향 |
|---|---|---|
| `strftime` (**F**ormat) | 날짜 객체 → 문자열 | 객체 → 출력 |
| `strptime` (**P**arse) | 문자열 → 날짜 객체 | 해석 → 객체 |

In [ ]:
now = datetime(2026, 4, 10, 14, 30, 0)

# strftime: 날짜 객체의 포맷 출력
print(now.strftime("%Y년 %m월 %d일"))       # 2026년 04월 10일
print(now.strftime("%Y-%m-%d %H:%M:%S"))
print(now.strftime("%A, %B %d, %Y"))

# strptime: 규칙에 맞춰 문자열을 해독
parsed = datetime.strptime("2026/04/10", "%Y/%m/%d")
print(f"파싱 결과: {parsed}  (type={type(parsed).__name__})")

### 📋 Cheat Sheet: 실무 날짜 포맷 코드

| 코드 | 의미 | 예시 |
|---|---|---|
| `%Y` | 4자리 연도 | 2026 |
| `%y` | 2자리 연도 | 26 |
| `%m` | 숫자 월 (01~12) | 04 |
| `%B` | 월 전체 영문 | April |
| `%d` | 숫자 일 (01~31) | 10 |
| `%A` | 요일 전체 영문 | Friday |
| `%a` | 요일 약어 | Fri |
| `%H` | 24시간제 시 (00~23) | 14 |
| `%I` | 12시간제 시 (01~12) | 02 |
| `%M` | 분 (00~59) | 30 |
| `%S` | 초 (00~59) | 00 |

In [ ]:
# Cheat Sheet 코드 전체 실습
sample = datetime(2026, 4, 10, 14, 30, 0)
codes = ["%Y", "%y", "%m", "%B", "%d", "%A", "%a", "%H", "%I", "%M", "%S"]
for c in codes:
    print(f"{c} → {sample.strftime(c)}")

---

# Part 2. Pandas 시계열 4대 코어 객체

대규모 데이터 처리를 위한 Pandas 시계열 객체:

| 객체 | 개념 | 특징 |
|---|---|---|
| `Timestamp` | 특정 순간(시점) | Python `datetime` 호환, 나노초(ns) 단위 고정밀 지원 |
| `DatetimeIndex` | `Timestamp`의 배열 집합 | `Series`/`DataFrame`의 색인 용도, 날짜 슬라이싱의 핵심 |
| `Timedelta` | 시간의 차이 (기간 길이) | `'2 days 03:00'` 형태의 문자열 파싱, 벡터화 산술 연산 |
| `Period` | 특정 시간 범위(구간) | 점이 아닌 `'2026-Q2'` 같은 면적(Range) 개념 |

## 2-1. `Timestamp`: 나노초 단위까지 통제하는 진화형 시점 객체

In [ ]:
# 생성 및 호환 — Pandas 내부 시스템은 모든 날짜를 기본적으로 Timestamp로 취급
ts = pd.Timestamp("2026-04-10 14:30:00")
ts_from_dt = pd.Timestamp(datetime(2026, 4, 10))

print(f"문자열 파싱 : {ts}")
print(f"datetime 변환: {ts_from_dt}")

# 강력한 자체 확장 메서드
print(f"요일 영문 : {ts.day_name()}")    # Friday
print(f"월 영문   : {ts.month_name()}")  # April
print(f"분기      : {ts.quarter}")        # 2
print(f"연중 주차 : {ts.week}")            # 15

## 2-2. `DatetimeIndex`: 데이터프레임 시계열 슬라이싱의 코어 엔진

- `pd.date_range()`를 통한 연속된 날짜 배열의 고속 생성
- 복잡한 조건문 없이 연도/월 단위 직관적 데이터 필터링 가능

In [ ]:
# 일별 데이터 10일치 인덱스 생성
dates = pd.date_range("2026-01-01", periods=10, freq="D")
ts_series = pd.Series(np.random.randn(10), index=dates)
print(ts_series)

In [ ]:
# 날짜 문자열 기반의 직관적 슬라이싱
print("◆ 2026-01-03 ~ 2026-01-07 구간")
print(ts_series["2026-01-03":"2026-01-07"])

In [ ]:
# 월/연도 단위 필터링 압축 지원
monthly_data = pd.Series(
    np.random.randn(365),
    index=pd.date_range("2026-01-01", periods=365, freq="D"),
)

print(f"3월 데이터 건수: {len(monthly_data['2026-03'])}")
print(f"2026년 전체 건수: {len(monthly_data['2026'])}")

## 2-3. `Timedelta`: 강력한 문자열 파싱과 대규모 벡터 산술 연산

In [ ]:
# 텍스트 직접 파싱으로 기간 객체 생성
td1 = pd.Timedelta("2 days 03:00:00")
td2 = pd.Timedelta("7 days")  # "1 week" 표기는 최신 pandas에서 미지원
print(f"td1 = {td1}")
print(f"td2 = {td2}")

# 시점 + 기간 = 새로운 미래 시점 연산
ts = pd.Timestamp("2026-04-10")
future = ts + pd.Timedelta("30 days")
print(f"30일 후: {future.date()}")  # 2026-05-10

# 두 시점 간의 차이 연산 (반환 타입: Timedelta)
diff = pd.Timestamp("2026-12-31") - ts
print(f"연말까지: {diff.days}일")

## 2-4. `Period`: 찰나의 '점(Point)'이 아닌 구간을 덮는 '선(Range)'

특정 시점이 아닌 *'2026년 4월 전체'* 혹은 *'2026년 2분기'* 등의 구간 자체를 하나의 단위로 취급합니다.

In [ ]:
p1 = pd.Period("2026-04", freq="M")   # 2026년 4월 블록
p3 = pd.Period("2026Q2", freq="Q")     # 2026년 2분기 블록

print(f"4월 시작 : {p1.start_time}")   # 2026-04-01 00:00:00
print(f"4월 종료 : {p1.end_time}")     # 2026-04-30 23:59:59...

# 논리적 기간 연산
print(f"p1 + 1   : {p1 + 1}")          # 2026-05 (다음 달로 점프)
print(f"p3 + 2   : {p3 + 2}")          # 2026Q4 (2개 분기 후로 점프)

## 2-5. 데이터 전처리 제1관문: 문자열 ↔ 날짜 양방향 고속 변환

- `pd.to_datetime()`: 단일 값부터 대용량 Series 전체까지 한 번에 형변환 수행
- `format` 파라미터 명시를 통한 파싱 속도 획기적 향상
- `errors='coerce'`: 잘못된 문자열을 안전하게 `NaT`(Not a Time)로 처리

In [ ]:
# 컬럼 전체를 날짜형으로 고속 변환
df = pd.DataFrame({"order_date": ["2026/01/01", "2026/02/15", "2026/03/20"]})
df["order_date"] = pd.to_datetime(df["order_date"], format="%Y/%m/%d")
print(df.dtypes)
print(df)

In [ ]:
# errors='coerce': 잘못된 문자열 처리
bad_dates = ["2026-01-01", "invalid", "2026-03-15"]
result = pd.to_datetime(bad_dates, errors="coerce")
print(result)
# 'invalid'는 안전하게 NaT(Not a Time) 처리됨

## 2-6. `.dt` 접근자: Series 데이터의 초고속 속성 분할 추출

- 수백만 행의 데이터에서 날짜 정보를 for 루프 없이 즉각적으로 추출하는 벡터화 연산 지원
- 실무 파생 변수(예: 주말 여부 필터링) 생성의 핵심 문법

In [ ]:
df = pd.DataFrame({
    "date_col": pd.date_range("2026-04-01", periods=10, freq="D")
})

# 연, 월, 일 독립 컬럼 고속 생성
df["year"] = df["date_col"].dt.year
df["month"] = df["date_col"].dt.month
df["day"] = df["date_col"].dt.day
df["day_name"] = df["date_col"].dt.day_name()

# 주말 여부 Boolean 파생 변수 (0: 월요일 ~ 6: 일요일)
df["is_weekend"] = df["date_col"].dt.dayofweek >= 5

print(df)

### 🧭 종합 Cheat Sheet: `.dt` 접근자 주요 속성

In [ ]:
s = pd.Series([pd.Timestamp("2026-04-10 14:30:00")])

print("[기본 요소 노드]")
print(f"  .year         → {s.dt.year.iloc[0]}")
print(f"  .month        → {s.dt.month.iloc[0]}")
print(f"  .day          → {s.dt.day.iloc[0]}")
print(f"  .hour         → {s.dt.hour.iloc[0]}")

print("\n[요일 및 분기 노드]")
print(f"  .dayofweek    → {s.dt.dayofweek.iloc[0]}  (4 = 금요일)")
print(f"  .day_name()   → {s.dt.day_name().iloc[0]}")
print(f"  .quarter      → {s.dt.quarter.iloc[0]}")

print("\n[비즈니스 로직 노드]")
print(f"  .is_month_end → {s.dt.is_month_end.iloc[0]}")
print(f"  .days_in_month→ {s.dt.days_in_month.iloc[0]}")

---

# Part 3. Pandas 시계열 특화 기능

- 기본 `datetime` 모듈의 한계 극복, 대규모 시계열 처리
- DataFrame 구조에 최적화된 시간 연산 지원
- 대량 데이터의 그룹화, 리샘플링, 결측치 처리 기능 제공

## 3-1. 시간 축의 생성: `pd.date_range()`

- 특정 시작일/종료일 기준 날짜 범위 자동 생성
- 시계열 분석용 더미 데이터 생성의 기반
- 데이터 리샘플링을 위한 기준점(Index) 역할 수행

### 기본 날짜 범위 및 빈도 설정

In [ ]:
# 1) 시작일/종료일 지정 방식
daily = pd.date_range("2026-01-01", "2026-01-10")
print("◆ 일별 범위 (start ~ end)")
print(daily)

# 2) 시작일과 개수(periods) 기반 생성 방식
weekly = pd.date_range("2026-01-01", periods=10, freq="W")
print("\n◆ 주별 범위 (10주)")
print(weekly)

### 다양한 빈도(Frequency) 옵션 적용

In [ ]:
# 시간별 (24시간)
hourly = pd.date_range("2026-01-01", periods=24, freq="h")
print(f"시간별 (앞 5개): {hourly[:5].tolist()}")

# 월초 기준 (12개월)
monthly = pd.date_range("2026-01-01", periods=12, freq="MS")
print(f"\n월초 (12개월):\n{monthly}")

# 영업일만 포함 (10일)
business = pd.date_range("2026-01-01", periods=10, freq="B")
print(f"\n영업일 (10일):\n{business}")

### 📋 Frequency 코드 Cheat Sheet

| 코드 | 의미 | 코드 | 의미 |
|---|---|---|---|
| `D` | 일 (Day) | `A` / `Y` | 연말 (Year end) |
| `W` | 주 (Week) | `h` | 시간 (Hour) |
| `M` / `ME` | 월말 (Month end) | `min` / `T` | 분 (Minute) |
| `MS` | 월초 (Month start) | `s` / `S` | 초 (Second) |
| `Q` / `QE` | 분기말 (Quarter end) | `B` | 영업일 (Business day) |

> 💡 Pandas 2.2 이후 일부 코드(`M`→`ME`, `H`→`h`, `Q`→`QE`)가 소문자/대체 표기로 변경되었습니다.

## 3-2. 시간 단위의 압축: `resample()`

- 시간 간격 재조정 및 데이터 그룹화 기능
- SQL의 `GROUP BY` 연산과 작동 원리 유사
- 일별 데이터를 월별 평균/합계 등으로 집계 시 활용

### 다운샘플링 (일별 → 월/주별 집계)

In [ ]:
# 90일치 가상 판매 데이터 생성 로직
dates = pd.date_range("2026-01-01", periods=90, freq="D")
df = pd.DataFrame(
    {"sales": np.random.randint(100, 500, 90)},
    index=dates,
)
print("◆ 원본 데이터 (앞 5행)")
print(df.head())

# 월별 누적 합계 (ME = Month End)
monthly_sum = df.resample("ME").sum()
print("\n◆ 월별 합계")
print(monthly_sum)

# 주별 평균
weekly_mean = df.resample("W").mean()
print("\n◆ 주별 평균 (앞 5행)")
print(weekly_mean.head().round(2))

### 다중 통계 적용 및 업샘플링(Upsampling)

- `agg()`: 여러 통계 지표 동시 산출
- **업샘플링**: 저빈도 데이터를 고빈도로 변환
- `ffill()`: 발생한 결측치를 이전 데이터로 보간(Forward Fill)

In [ ]:
# 여러 통계를 한 번에 적용
monthly_stats = df.resample("ME").agg(["sum", "mean", "max", "min"])
print("◆ 월별 다중 통계")
print(monthly_stats.round(2))

In [ ]:
# 업샘플링 (월별 → 일별 변환 및 앞값 채우기)
daily_from_monthly = monthly_sum.resample("D").ffill()
print("◆ 월별 → 일별 업샘플링 (앞 10행)")
print(daily_from_monthly.head(10))

## 3-3. 시간축 이동: `shift()`

- 데이터를 시간축 방향으로 밀거나 당기는 기능
- **Lag(지연) 피처 생성의 핵심 메커니즘**
- 전일 대비 수익률, 증감률 계산 등 금융/시계열 분석 필수 함수

In [ ]:
# 금융 데이터 응용: 전일 대비 수익률 산출
df = pd.DataFrame(
    {"price": [100, 102, 98, 105]},
    index=pd.date_range("2026-04-01", periods=4),
)

# 전일 종가 파생 변수 (shift(1): 한 행 아래로 이동)
df["prev_price"] = df["price"].shift(1)

# 전일 대비 수익률 계산
df["return_pct"] = (df["price"] / df["price"].shift(1) - 1) * 100

print(df.round(2))

## 3-4. 고급 응용: 이동평균 (Moving Average) 편차 계산

- `rolling(window).mean()`: 특정 기간의 이동평균선 생성
- 현재 주가와 3일 이동평균선 간의 차이(`diff_from_ma`) 계산

In [ ]:
# 10일치 가상 종가 데이터
df = pd.DataFrame(
    {"price": [100, 102, 98, 105, 108, 106, 110, 112, 109, 115]},
    index=pd.date_range("2026-04-01", periods=10),
)

# 3일 이동평균선 계산
df["ma3"] = df["price"].rolling(3).mean()

# 이동평균 대비 현재가 차이
df["diff_from_ma"] = df["price"] - df["ma3"]

print(df.round(2))

---

# Part 4. `dateutil.relativedelta` — 인간 친화적 의미 연산

| 항목 | `timedelta` | `relativedelta` |
|---|---|---|
| 정의 | 절대적 기간 | 상대적 기간 |
| 단위 | `days`, `seconds` 단위로 고정 | `years`, `months` 단위 지원 |
| 특징 | '30일 후' 계산 가능 / '1개월 후' 계산 불가 | 월마다 다른 일수 자동 처리 |
| 윤년 | 수동 처리 필요 | 윤년(Leap year) 발생 시 자동 고려 |

> 💡 실무 시계열 분석에서는 비즈니스 로직에 맞춰 두 가지 기간 연산 방식을 정확히 구분하여 적용해야 합니다.

## 4-1. 기본 연산: 월/년 단위 상대적 계산

In [ ]:
now = datetime(2026, 4, 10)

# 1개월 후 (timedelta로는 불가)
next_month = now + relativedelta(months=1)
print(f"1개월 후   : {next_month.date()}")  # 2026-05-10

# 2년 후
two_years = now + relativedelta(years=2)
print(f"2년 후     : {two_years.date()}")   # 2028-04-10

# 복합 연산: 1년 2개월 3일 후
future = now + relativedelta(years=1, months=2, days=3)
print(f"복합 연산  : {future.date()}")

## 4-2. 고급 응용: 의미적 날짜 산출 및 만 나이 계산

- 특정 달의 며칠이든 무관하게 **'말일'** 자동 스냅핑 로직
- 과거 날짜(생년월일) 기반 정확한 **만 나이** 산출 (윤년/일수 자동 고려)

In [ ]:
now = datetime(2026, 4, 10)

# '다음 달 말일' 계산 — day=31을 넘겨도 해당 월의 말일로 자동 스냅
last_day_next_month = now + relativedelta(months=1, day=31)
print(f"다음 달 말일: {last_day_next_month.date()}")  # 2026-05-31

# 2월의 경우에도 자동으로 28 또는 29일로 보정
feb_last = datetime(2026, 1, 15) + relativedelta(months=1, day=31)
print(f"2026년 2월 말일: {feb_last.date()}")           # 2026-02-28

In [ ]:
# 생년월일 기반 정확한 만 나이 계산
birth = datetime(1990, 5, 15)
age = relativedelta(now, birth)
print(f"나이: {age.years}세 {age.months}개월 {age.days}일")

## 4-3. [종합 실습] `timedelta` vs `relativedelta` 직접 비교

동일한 *'1개월 후'* 요구사항을 두 방식으로 처리했을 때 결과가 어떻게 달라지는지 확인합니다.

In [ ]:
test_dates = [
    datetime(2026, 1, 31),   # 1월 31일
    datetime(2026, 2, 15),   # 2월 15일 (평년)
    datetime(2024, 2, 29),   # 윤년 2월 29일
    datetime(2026, 3, 31),   # 3월 31일
]

print(f"{'원본 날짜':<14} | {'+30일 (timedelta)':<22} | {'+1개월 (relativedelta)':<22}")
print("-" * 66)
for d in test_dates:
    td_result = (d + timedelta(days=30)).date()
    rd_result = (d + relativedelta(months=1)).date()
    print(f"{str(d.date()):<14} | {str(td_result):<22} | {str(rd_result):<22}")

---

## ✅ 학습 체크리스트

- [ ] Python 내장 `datetime` 모듈의 4대 클래스를 구분할 수 있다
- [ ] `timedelta`의 한계(월/년 연산 불가)를 이해하고 있다
- [ ] `strftime`과 `strptime`의 방향을 혼동하지 않는다
- [ ] Pandas `Timestamp`, `DatetimeIndex`, `Timedelta`, `Period`의 차이를 설명할 수 있다
- [ ] `pd.to_datetime(errors='coerce')`로 더티 데이터를 안전하게 처리할 수 있다
- [ ] `.dt` 접근자로 벡터화된 파생 변수를 생성할 수 있다
- [ ] `resample()`로 다운/업샘플링을 수행할 수 있다
- [ ] `shift()`로 Lag 피처 및 수익률을 계산할 수 있다
- [ ] `rolling()`으로 이동평균을 산출할 수 있다
- [ ] `relativedelta`로 '다음 달 말일', '만 나이' 등 의미적 연산을 수행할 수 있다
